# Schema-poverty ablation

**Purpose**: manufacture controlled headroom on a fixed dataset by degrading target-record attribute richness in principled steps. Richness becomes the only moving variable; positive rate, LLM demonstrations, and all other factors are held constant across rungs. Delivers the paper's most novel positive result, the paradigm gap widens toward the sparse end of the ladder plus the headroom re-analysis (was demo-selection helping at 8B because of headroom, not backbone size?).

**Prerequisite**: run `python -m src.analysis.field_discriminativeness` first to produce `results/schema_poverty_ablation/field_discriminativeness.csv`. That CSV determines the drop order per dataset (ascending MI = least-informative-first).

**Scope of this notebook**: LLM paradigm only (random_k2 and cider_k2 on LLaMA-3.3-70B via DeepInfra). Ditto warm-start re-runs at each rung are a separate GPU-heavy notebook (`the SFT ablation notebook`, TBD).

**Output**: `results/runs/sparsity-ablation/<pair>/<rung>/<perturbation>/<method>/seed_<N>/metrics.json`

## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/cd-er-paradigm-choice'
os.chdir(BASE)
os.environ['REPO_ROOT'] = BASE
sys.path.insert(0, BASE)

In [ ]:
!pip install -q rapidfuzz
from rapidfuzz.distance import Levenshtein as _rf_Lev, JaroWinkler as _rf_JW
import sys, types

_lev = types.ModuleType('Levenshtein')
_lev.distance = _rf_Lev.distance
_lev.ratio = lambda a, b: _rf_Lev.normalized_similarity(a, b)
sys.modules['Levenshtein'] = _lev

_jf = types.ModuleType('jellyfish')
_jf.jaro_winkler_similarity = lambda a, b: _rf_JW.normalized_similarity(a, b)
sys.modules['jellyfish'] = _jf

print('[ok] Levenshtein + jellyfish shims installed via rapidfuzz')


## 2. Config: the sparsity matrix

Two pairs, five rungs (dropped by MI order from `field_discriminativeness.csv`), five perturbation variants (`none` + 4 contamination probes), two demo-selection methods, three seeds, one backbone (LLaMA-3.3-70B is where the ceiling effect lives; running the headroom re-analysis on this backbone directly tests whether demosntration selection was a capability effect or a headroom effect).

Full matrix: 2 pairs × 5 rungs × 5 perturbations × 2 methods × 3 seeds = 300 cells. Perturbations 1–4 (short-form, glued, initials, shuffle) only run at rungs 3 and 4 (sparse end) to keep cost bounded; unperturbed (rung × method × seed) runs at all 5 rungs. Effective count ≈ 2 × [5 (unperturbed) + 4 × 2 (perturbations at sparse rungs)] × 2 × 3 = 156 cells.

In [ ]:
SUBSET = 'phase1'    # 'phase1' | 'contamination' | 'all'

# One backbone for schema-poverty ablation: LLaMA-3.3-70B is where the ceiling effect lives.
BACKBONES = {
    'llama-3.3-70b': {
        'provider': 'deepinfra',
        'model': 'meta-llama/Llama-3.3-70B-Instruct',
    },
}

# Two pairs so the effect is not regime-specific.
# The exact rung field-sets are derived in cell 9 from field_discriminativeness.csv.
PAIRS = [
    ('Structured/Walmart-Amazon', 'Structured/Amazon-Google'),   # product (headline)
    ('Structured/Walmart-Amazon', 'Structured/DBLP-ACM'),        # citation (generality)
    ('Structured/Walmart-Amazon', 'Textual/Abt-Buy'),            # textual product (regime diversity)
]

# --- Mechanism probe: citation-source demos on the LLM side ---
# Default is None: demos are sampled from the source in each PAIRS tuple
# (Walmart-Amazon for all three targets).
#
# To run the mechanism-isolation probe that tests whether the DBLP-ACM
# sparse-rung inversion is caused by the LLM's product-domain demonstrations,
# uncomment the DA -> DA override below. This reruns DA rungs 3 and 4 only
# with demonstrations sampled from the DBLP-ACM train split instead of the
# Walmart-Amazon train split. Cost estimate: ~$0.30 on DeepInfra.
#
# Results write to `results/runs/sparsity-ablation/<backbone>/<target>__demos_<source>/...`
# so they do not overwrite the baseline WA-source runs.
DEMO_SOURCE_OVERRIDES = {
    # target_ds -> demo_source_ds
    'Structured/DBLP-ACM': 'Structured/DBLP-ACM',  # citation demos for the mechanism probe
}
PROBE_RUNGS = [3, 4]     # rungs at which to run the mechanism probe (irrelevant if empty)
PROBE_PERTURBATIONS = ['none', 'glued', 'short_form']

METHODS = ['random_k2']

SEEDS = [42, 123, 456]

# Rungs are numeric labels; the concrete field-set for each rung is loaded from
# field_discriminativeness.csv (drop least-informative-first).
RUNGS = [0, 1, 2, 3, 4]

# Contamination perturbations applied at sparse rungs only (3 and 4).
# 'none' means unperturbed and runs at every rung.
PERTURBATIONS = ['none', 'glued', 'short_form']
# initials + shuffle deferred: glued and short_form are the strongest
# merchant-format contamination probes. Add back for a follow-on.
# PERTURBATIONS = ['none', 'short_form', 'glued', 'initials', 'shuffle']
PERTURBATION_RUNGS = {
    'none': RUNGS,
    'glued': [3, 4],
    'short_form': [3, 4],
    # 'initials': [3, 4],   # deferred
    # 'shuffle':  [3, 4],   # deferred
}

# CIDER hparams (paper defaults, from main matrix runner).
GAMMA = 0.01
ALPHA = 0.5
H_CANDIDATES = 50
K_DEMOS = 2
KFOLDS = 5

SKIP_IF_DONE = True
MAX_BUDGET_USD = 15.0

print(f'Running SUBSET={SUBSET}')
print(f'  backbones: {list(BACKBONES.keys())}')
print(f'  pairs: {len(PAIRS)} pairs')
print(f'  rungs: {RUNGS}')
print(f'  methods: {METHODS}')
print(f'  perturbations: {PERTURBATIONS}')
print(f'  seeds: {SEEDS}')
n_unperturbed = len(PAIRS) * len(RUNGS) * len(METHODS) * len(SEEDS)
n_contam = len(PAIRS) * 2 * (len(PERTURBATIONS) - 1) * len(METHODS) * len(SEEDS)
print(f'  cells: unperturbed={n_unperturbed}  contamination={n_contam}  total={n_unperturbed + n_contam}')

## 3. API client (DeepInfra)

In [ ]:
from google.colab import userdata
from openai import OpenAI

PROVIDER_ENDPOINTS = {
    'together':  ('TOGETHER_API_KEY',  'https://api.together.xyz/v1'),
    'deepinfra': ('DEEPINFRA_API_KEY', 'https://api.deepinfra.com/v1/openai'),
    'openai':    ('OPENAI_API_KEY',    None),
}

def make_client(cfg):
    provider = cfg['provider']
    secret_key, base_url = PROVIDER_ENDPOINTS[provider]
    api_key = userdata.get(secret_key)
    if not api_key:
        raise RuntimeError(f'Missing Colab secret {secret_key!r} for provider {provider!r}.')
    kwargs = {'api_key': api_key}
    if base_url is not None:
        kwargs['base_url'] = base_url
    return OpenAI(**kwargs), cfg['model']

CLIENTS = {name: make_client(cfg) for name, cfg in BACKBONES.items()}
for name, (client, model) in CLIENTS.items():
    print(f'  {name}: {model}')

## 4. Load MI ranking → derive per-rung field sets

In [ ]:
import pandas as pd
from pathlib import Path as _P

mi_path = _P(BASE) / 'results' / 'schema_poverty_ablation' / 'field_discriminativeness.csv'
assert mi_path.exists(), (
    f'MI ranking CSV not found at {mi_path}. Run `python -m src.analysis.field_discriminativeness` first.'
)
MI_DF = pd.read_csv(mi_path)
print(MI_DF.to_string(index=False))


def rung_field_set(target_dataset, rung):
    """Return the set of field names retained at the given rung for the target.

    Rung 0 = all fields. Rung 1 drops the lowest-MI field. Rung 2 drops the two
    lowest. Rung 3 drops all but the highest-MI field. Rung 4 also truncates the
    remaining field's value to the first 3 tokens (merchant-name stub).
    """
    sub = MI_DF[MI_DF['dataset'] == target_dataset].sort_values('mi', ascending=True)
    if sub.empty:
        raise ValueError(f'No MI rows found for {target_dataset}')
    fields_asc_by_mi = sub['field'].tolist()
    fields_desc_by_mi = list(reversed(fields_asc_by_mi))
    if rung == 0:
        return set(fields_asc_by_mi), False
    if rung == 1:
        return set(fields_asc_by_mi[1:]), False
    if rung == 2:
        return set(fields_asc_by_mi[2:]), False
    if rung == 3:
        return {fields_desc_by_mi[0]}, False
    if rung == 4:
        return {fields_desc_by_mi[0]}, True     # (single top field, truncate=True)
    raise ValueError(rung)


# Show the ladder for each target for the paper table
print('\nLadder per target:')
for _s, tgt in PAIRS:
    print(f'  {tgt}:')
    for r in RUNGS:
        fs, trunc = rung_field_set(tgt, r)
        print(f'    rung {r}: {sorted(fs)}  truncate={trunc}')

## 5. CIDER-helpers helpers (inlined for self-containment)

In [ ]:
# sys.path insurance
import sys, os
_BASE = '/content/drive/MyDrive/cd-er-paradigm-choice'
if _BASE not in sys.path:
    sys.path.insert(0, _BASE)
if os.path.isdir(_BASE) and os.getcwd() != _BASE:
    os.chdir(_BASE)
os.environ.setdefault('REPO_ROOT', _BASE)

SOURCE_FOR_TARGET = {
    'Textual/Abt-Buy':          'Structured/Walmart-Amazon',
    'wdc/watches':              'wdc/computers',
    'Structured/DBLP-ACM':      'Structured/Walmart-Amazon',
    'Structured/Amazon-Google': 'Structured/Walmart-Amazon',
    'Dirty/DBLP-ACM':           'Structured/Walmart-Amazon',
}

DOMAIN_INFO = {
    'Textual/Abt-Buy':          'product',
    'wdc/watches':              'watch',
    'wdc/computers':            'computer',
    'Structured/DBLP-ACM':      'publication',
    'Structured/Amazon-Google': 'software product',
    'Structured/Walmart-Amazon':'product',
    'Dirty/DBLP-ACM':           'publication',
}

TARGETS = [t for _s, t in PAIRS]
SBERT_MODEL = 'sentence-transformers/all-mpnet-base-v2'
PROVIDER = 'deepinfra'

# Load helpers from CIDER baseline probe by direct exec — reuses the same pipeline main matrix uses.
# The helpers define: load_ditto_split, get_source_train_and_target_test,
# encode_entity_pairs, structural_vectors_for_pairs, select_candidate_source,
# cider_similarity_matrix, select_demos_for_targets, build_cider_prompt,
# parse_yes_no, compute_metrics.
import json as _json
from pathlib import Path as __P
_baseline_nb = _json.loads(__P('notebooks/(CIDER baseline reference)').read_text())
_helper_cells = [9, 11, 13, 15, 17, 19, 21, 24]
for _i in _helper_cells:
    _src = ''.join(_baseline_nb['cells'][_i].get('source', []))
    if _i == 21:
        # cell 21 has llm_call AND parse_yes_no — keep only parse_yes_no
        _pi = _src.find('def parse_yes_no')
        if _pi >= 0:
            _src = _src[_pi:]
    exec(_src, globals())

print('[ok] CIDER baseline probe helpers inlined via exec')

## 6. Backbone-parameterised LLM call (with token/latency instrumentation)

In [ ]:
import time

def llm_call_with(client, model, prompt, max_retries=3, max_tokens=10):
    """Call chat.completions with usage/latency; returns (text, prompt_tokens, completion_tokens, latency_sec)."""
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=max_tokens,
                temperature=0.0,
            )
            latency = time.perf_counter() - t0
            usage = getattr(resp, 'usage', None)
            pt = getattr(usage, 'prompt_tokens', 0) if usage else 0
            ct = getattr(usage, 'completion_tokens', 0) if usage else 0
            return resp.choices[0].message.content.strip(), pt, ct, latency
        except Exception as e:
            if attempt == max_retries - 1:
                return '', 0, 0, 0.0
            time.sleep(2 ** attempt)
    return '', 0, 0, 0.0

## 7. Field-masking and contamination-perturbation functions

The helpers here transform records before prompt construction (LLM paradigm) or before serialisation (SFT paradigm). Same functions applied to both records in a test pair; demos are always at full schema (Rung 0).

In [ ]:
import re
import random as _rand

# Match one COL <name> VAL <value> segment (see field_discriminativeness.py).
_FIELD_RE = re.compile(r'COL\s+(\S+)\s+VAL\s*(.*?)\s*(?=COL\s+|$)')


def parse_record(rec_str):
    """Return list of (field_name, value) pairs from a Ditto COL/VAL string."""
    return [(m.group(1), m.group(2).strip()) for m in _FIELD_RE.finditer(rec_str)]


def format_record(fields):
    """Inverse of parse_record."""
    return ' '.join(f'COL {n} VAL {v}' for n, v in fields)


def mask_record(rec_str, keep_fields, truncate_title_to_3=False):
    fields = parse_record(rec_str)
    kept = [(n, v) for n, v in fields if n in keep_fields]
    if truncate_title_to_3:
        # At the truncated stub rung, exactly one field is retained (the top-MI
        # field). Truncate whichever field it is — the previous 'n == "title"'
        # guard was a latent bug for Abt-Buy (top field 'name') and Walmart-Amazon
        # (top field 'modelno').
        kept = [(n, ' '.join(v.split()[:3])) for n, v in kept]
    return format_record(kept)


def mask_pair(pair, keep_fields, truncate_title_to_3=False):
    return {
        'left':  mask_record(pair['left'],  keep_fields, truncate_title_to_3),
        'right': mask_record(pair['right'], keep_fields, truncate_title_to_3),
        'label': pair['label'],
    }


# ---------------------------------------------------------------------------
# Contamination perturbations (applied to values only; field names preserved).
# ---------------------------------------------------------------------------

_ABBREV = {
    'corporation': 'corp', 'company': 'co', 'limited': 'ltd', 'incorporated': 'inc',
    'international': 'intl', 'brothers': 'bros', 'systems': 'sys',
    'street': 'st', 'avenue': 'ave', 'road': 'rd', 'boulevard': 'blvd',
    'united states': 'us', 'university': 'univ',
}


def _short_form_value(v):
    tokens = v.split()
    return ' '.join(_ABBREV.get(t.lower(), t) for t in tokens)


def _glued_value(v):
    return re.sub(r'[^A-Za-z0-9]+', '', v).upper()


def _initials_value(v):
    tokens = v.split()
    if len(tokens) < 2:
        return v
    return '.'.join(t[0].upper() for t in tokens if t) + '.'


def _shuffle_value(v, seed):
    tokens = v.split()
    r = _rand.Random(seed)
    r.shuffle(tokens)
    return ' '.join(tokens)


def perturb_record(rec_str, perturbation, seed):
    if perturbation == 'none':
        return rec_str
    fields = parse_record(rec_str)
    fn = {
        'short_form': _short_form_value,
        'glued':      _glued_value,
        'initials':   _initials_value,
        'shuffle':    lambda v: _shuffle_value(v, seed),
    }[perturbation]
    fields = [(n, fn(v) if v else v) for n, v in fields]
    return format_record(fields)


def perturb_pair(pair, perturbation, seed):
    return {
        'left':  perturb_record(pair['left'],  perturbation, seed),
        'right': perturb_record(pair['right'], perturbation, seed + 1),
        'label': pair['label'],
    }


# Sanity check
_sample = 'COL title VAL McDonald\'s Restaurant #4521 COL brand VAL McDonalds COL price VAL 5.99'
print('original :', _sample)
print('short    :', perturb_record(_sample, 'short_form', 42))
print('glued    :', perturb_record(_sample, 'glued', 42))
print('initials :', perturb_record(_sample, 'initials', 42))
print('shuffle  :', perturb_record(_sample, 'shuffle', 42))
print('rung 3   :', mask_record(_sample, {'title'}))
print('rung 4   :', mask_record(_sample, {'title'}, truncate_title_to_3=True))

## 8. Selection functions

In [ ]:
import numpy as np

def select_random_stratified_demos(source_pairs, k=2, seed=42):
    r = _rand.Random(seed)
    pos = [p for p in source_pairs if p['label'] == 1]
    neg = [p for p in source_pairs if p['label'] == 0]
    n_pos, n_neg = k // 2, k - k // 2
    return (r.sample(pos, min(n_pos, len(pos))) + r.sample(neg, min(n_neg, len(neg))))

## 9. SBERT cache (from main matrix SBERT optimization)

In [ ]:
from pathlib import Path as __P

_SBERT_CACHE = {}
_SBERT_CACHE_DIR = __P(BASE) / 'results' / 'cache' / 'sbert'
_SBERT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _get_sbert_features(source_train, target_test, source_ds, target_ds, cache_suffix=''):
    """Cached SBERT encoding keyed by (source_ds, target_ds, cache_suffix).

    cache_suffix distinguishes different field-masking / perturbation variants of
    the same underlying pair.
    """
    key = (source_ds, target_ds, cache_suffix)
    if key in _SBERT_CACHE:
        return _SBERT_CACHE[key]
    tag = source_ds.replace('/', '__') + '__' + target_ds.replace('/', '__')
    if cache_suffix:
        tag += '__' + cache_suffix
    cache_file = _SBERT_CACHE_DIR / (tag + '.npz')
    if cache_file.exists():
        d = np.load(cache_file)
        pair = (d['src_sem'], d['tgt_sem'])
        _SBERT_CACHE[key] = pair
        return pair
    src_sem = encode_entity_pairs(source_train)
    tgt_sem = encode_entity_pairs(target_test)
    _SBERT_CACHE[key] = (src_sem, tgt_sem)
    np.savez(cache_file, src_sem=src_sem, tgt_sem=tgt_sem)
    return src_sem, tgt_sem

## 10. Per-cell runner

In [ ]:
import json
from pathlib import Path

OUT_BASE = Path(BASE) / 'results' / 'runs' / 'sparsity-ablation'


def run_one_cell(backbone_name, source_ds, target_ds, method, seed, rung, perturbation, demo_source_ds=None):
    pair_tag = target_ds.replace('/', '__')
    # If demo_source_ds is set and differs from the default (source_ds), tag the output
    # path so mechanism-probe runs do not overwrite the baseline WA-source runs.
    if demo_source_ds is not None and demo_source_ds != source_ds:
        pair_tag = pair_tag + '__demos_' + demo_source_ds.replace('/', '__')
    out_dir = (OUT_BASE / backbone_name / pair_tag /
               f'rung_{rung}' / f'perturb_{perturbation}' / method / f'seed_{seed}')
    metrics_path = out_dir / 'metrics.json'
    if SKIP_IF_DONE and metrics_path.exists():
        existing = json.load(open(metrics_path))
        if existing.get('test_f1') is not None and existing.get('returncode') == 0:
            print(f'  [skip] rung={rung} perturb={perturbation} {method} seed_{seed}: F1={existing["test_f1"]:.4f}')
            return existing

    # Rung-dedup guard: skip rungs whose field set is identical to the previous
    # rung's (happens for 3-field targets where rung 3 = rung 2 = {top field only}).
    _kf_this, _tr_this = rung_field_set(target_ds, rung)
    if rung > 0:
        _kf_prev, _tr_prev = rung_field_set(target_ds, rung - 1)
        if _kf_this == _kf_prev and _tr_this == _tr_prev:
            print(f'  [rung dedup] rung {rung} identical to rung {rung-1} for {target_ds}, skipping')
            return None

    # Load full-schema data
    global SOURCE_FOR_TARGET
    if target_ds not in SOURCE_FOR_TARGET:
        SOURCE_FOR_TARGET[target_ds] = source_ds
    # Load target test data (target_test_full is always from target_ds)
    source_train, target_test_full, resolved_source = get_source_train_and_target_test(target_ds)
    # Override demo pool if a different source is requested for the mechanism probe.
    if demo_source_ds is not None and demo_source_ds != resolved_source:
        _saved_map = SOURCE_FOR_TARGET.get(demo_source_ds)
        SOURCE_FOR_TARGET[demo_source_ds] = demo_source_ds  # trigger loading of demo_source's train
        source_train, _, resolved_source = get_source_train_and_target_test(demo_source_ds)
        if _saved_map is not None:
            SOURCE_FOR_TARGET[demo_source_ds] = _saved_map

    # Field mask target test pairs at this rung
    keep_fields, truncate_title = rung_field_set(target_ds, rung)
    target_test = [mask_pair(p, keep_fields, truncate_title) for p in target_test_full]

    # Apply perturbation if any
    if perturbation != 'none':
        target_test = [perturb_pair(p, perturbation, seed) for p in target_test]

    # Demonstrations always at full schema (source, unperturbed)
    n_test = len(target_test)
    pos_rate = sum(p['label'] for p in target_test) / n_test
    src_dom = DOMAIN_INFO.get(resolved_source, 'entity')
    tgt_dom = DOMAIN_INFO.get(target_ds, 'entity')

    t_start = time.time()
    print(f'\n=== rung={rung} perturb={perturbation} {method} seed_{seed} ({target_ds}) ===')

    if method == 'random_k2':
        demos_fixed = select_random_stratified_demos(source_train, k=2, seed=seed)
        demo_getter = lambda i: demos_fixed
        src_sem, tgt_sem = None, None
    elif method == 'cider_k2':
        cache_sfx = f'rung{rung}_{perturbation}'
        src_sem, tgt_sem = _get_sbert_features(source_train, target_test, resolved_source, target_ds, cache_sfx)
        candidate_idx, _ = select_candidate_source(source_train, src_sem, tgt_sem,
                                                     h=H_CANDIDATES, gamma=GAMMA)
        candidate_pairs = [source_train[j] for j in candidate_idx]
        candidate_sem = src_sem[candidate_idx]
        candidate_struct = structural_vectors_for_pairs(candidate_pairs)
        target_struct = structural_vectors_for_pairs(target_test)
        sim_matrix = cider_similarity_matrix(tgt_sem, candidate_sem,
                                              target_struct, candidate_struct, alpha=ALPHA)
        candidate_labels = [p['label'] for p in candidate_pairs]
        demo_indices = select_demos_for_targets(sim_matrix, k=K_DEMOS,
                                                  candidate_labels=candidate_labels)
        demo_getter = lambda i: [candidate_pairs[j] for j in demo_indices[i]]
    else:
        raise ValueError(method)

    client, model = CLIENTS[backbone_name]
    preds, labels = [], []
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_api_latency_sec = 0.0
    for i, target_pair in enumerate(target_test):
        demos = demo_getter(i)
        prompt = build_cider_prompt(target_pair, demos, src_dom, tgt_dom)
        resp, pt, ct, lat = llm_call_with(client, model, prompt)
        preds.append(parse_yes_no(resp))
        labels.append(target_pair['label'])
        total_prompt_tokens += pt
        total_completion_tokens += ct
        total_api_latency_sec += lat
        if (i + 1) % 500 == 0:
            f1t, _, _ = compute_metrics(preds, labels)
            print(f'    [{i+1}/{n_test}] running F1={f1t:.4f}')

    f1, p, r = compute_metrics(preds, labels)
    elapsed = time.time() - t_start
    print(f'  [done] F1={f1:.4f}, P={p:.4f}, R={r:.4f}, elapsed={elapsed:.0f}s')

    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / 'predictions.jsonl', 'w') as f:
        for i, (pd_, lb) in enumerate(zip(preds, labels)):
            f.write(json.dumps({'i': i, 'pred': pd_, 'label': lb}) + '\n')

    payload = {
        'method': method,
        'backbone': backbone_name,
        'model': model,
        'source_dataset': source_ds,
        'target_dataset': target_ds,
        'rung': rung,
        'perturbation': perturbation,
        'kept_fields': sorted(keep_fields),
        'title_truncated': truncate_title,
        'seed': seed,
        'n_test_pairs': n_test,
        'positive_rate': pos_rate,
        'elapsed_sec': elapsed,
        'api_latency_sec': total_api_latency_sec,
        'prompt_tokens_total': total_prompt_tokens,
        'completion_tokens_total': total_completion_tokens,
        'prompt_tokens_mean': total_prompt_tokens / max(n_test, 1),
        'completion_tokens_mean': total_completion_tokens / max(n_test, 1),
        'test_f1': f1,
        'test_precision': p,
        'test_recall': r,
        'returncode': 0,
    }
    (out_dir / 'metrics.json').write_text(json.dumps(payload, indent=2))
    print(f'  [wrote] {out_dir}/metrics.json')
    return payload

## 11. Main sparsity ablation loop

In [ ]:
PRICING_USD_PER_MTOKEN = {
    'meta-llama/Llama-3.3-70B-Instruct':         (0.23, 0.40),
    'meta-llama/Meta-Llama-3.1-8B-Instruct':     (0.03, 0.06),
    'Qwen/Qwen2.5-72B-Instruct':                 (0.35, 0.40),
}


def _cell_cost_usd(payload):
    if not payload:
        return 0.0
    price = PRICING_USD_PER_MTOKEN.get(payload.get('model'))
    if not price:
        return 0.0
    in_p, out_p = price
    return (payload.get('prompt_tokens_total', 0) * in_p + payload.get('completion_tokens_total', 0) * out_p) / 1_000_000


all_results = []
running_cost_usd = 0.0
budget_hit = False

# Build the list of (source_ds, target_ds, demo_source_ds) tuples. Default: demo_source_ds = source_ds
# for every pair. If DEMO_SOURCE_OVERRIDES contains an entry for a target, also add a probe run.
_probe_runs = []
for _tgt, _demo_src in DEMO_SOURCE_OVERRIDES.items():
    _probe_runs.append((SOURCE_FOR_TARGET.get(_tgt, 'Structured/Walmart-Amazon'), _tgt, _demo_src))
if _probe_runs:
    print(f'[probe] {len(_probe_runs)} mechanism-probe target(s) queued with alternate demo source')

for backbone_name in BACKBONES:
    if budget_hit: break
    # First pass: baseline PAIRS with default (source_ds) demo pool
    for (source_ds, target_ds) in PAIRS:
        if budget_hit: break
        for rung in RUNGS:
            if budget_hit: break
            for perturbation in PERTURBATIONS:
                if budget_hit: break
                if rung not in PERTURBATION_RUNGS[perturbation]:
                    continue
                for method in METHODS:
                    if budget_hit: break
                    for seed in SEEDS:
                        if running_cost_usd > MAX_BUDGET_USD:
                            print(f'\n[BUDGET CAP] cumulative ${running_cost_usd:.2f} > ${MAX_BUDGET_USD:.2f} — halting.')
                            budget_hit = True
                            break
                        try:
                            r = run_one_cell(backbone_name, source_ds, target_ds, method, seed, rung, perturbation, demo_source_ds=None)
                            if r is None:
                                continue   # rung dedup — nothing was run
                            all_results.append(r)
                            running_cost_usd += _cell_cost_usd(r)
                            print(f'  [$ total = ${running_cost_usd:.2f} / ${MAX_BUDGET_USD}]')
                        except Exception as e:
                            print(f'[ERR] rung={rung} perturb={perturbation} {method} seed_{seed} ({target_ds}): {e}')
                            import traceback; traceback.print_exc()

    # Second pass: mechanism-probe runs with alternate demo source, restricted to PROBE_RUNGS/PROBE_PERTURBATIONS.
    for (source_ds, target_ds, demo_source_ds) in _probe_runs:
        if budget_hit: break
        for rung in PROBE_RUNGS:
            if budget_hit: break
            for perturbation in PROBE_PERTURBATIONS:
                if budget_hit: break
                for method in METHODS:
                    if budget_hit: break
                    for seed in SEEDS:
                        if running_cost_usd > MAX_BUDGET_USD:
                            print(f'\n[BUDGET CAP] cumulative ${running_cost_usd:.2f} > ${MAX_BUDGET_USD:.2f} — halting.')
                            budget_hit = True; break
                        try:
                            r = run_one_cell(backbone_name, source_ds, target_ds, method, seed, rung, perturbation, demo_source_ds=demo_source_ds)
                            if r is None: continue
                            all_results.append(r)
                            running_cost_usd += _cell_cost_usd(r)
                            print(f'  [probe $ total = ${running_cost_usd:.2f} / ${MAX_BUDGET_USD}]')
                        except Exception as e:
                            print(f'[ERR probe] rung={rung} perturb={perturbation} {method} seed_{seed} ({target_ds} <- {demo_source_ds}): {e}')
                            import traceback; traceback.print_exc()

print(f'\n[complete] cells run: {len(all_results)}, total cost ${running_cost_usd:.2f}')
if budget_hit:
    print('(halted at budget cap; raise MAX_BUDGET_USD and rerun — completed cells skipped)')

## 12. Summary

In [ ]:
print(f"{'target':<30} {'rung':>4} {'perturb':<10} {'method':<12} {'seed':>5} {'F1':>7}")
for r in all_results:
    if not r:
        continue
    print(f"{r['target_dataset']:<30} {r['rung']:>4} {r['perturbation']:<10} {r['method']:<12} {r['seed']:>5} {r['test_f1']:>7.4f}")